# Proyecto Final — Módulo 6

## Tema: Población adulta estadounidense (1994) con ingreso anual > $50K USD

| **Campo** | **Detalle** |
|---|---|
| **Grupo** | 2 |
| **Integrantes** | Cinthia Montero, Sebastián Calvo |
| **Curso** | Ciencia de Datos |
| **Fecha de Entrega** | 04 Setiembre 2026 |

---


# Problema:  
# Averigüar las características de la poblacióna adulta estadounidense del año 1994 que tenía ingresos anuales mayores a $ 50K USD y predecir sus futuros comportamientos.

# Objetivo:
# Identificar características que sean claves para determinar patrones que desarrollen individuos de la población que generen un ingreso anual mayor a $ 50K USD.

# **ETAPAS :**

# 1. Ingesta, validación, limpieza y transformación

## 1.1 Extracción de datos y visualización de características 

In [2]:
# Librerías para manipulación y análisis de datos

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Permite visualizar todas las columnas del DataFrame
pd.set_option('display.max_columns', None)

import os
os.environ["OMP_NUM_THREADS"] = "5"

In [4]:
# Instalamos 

!pip install ucimlrepo


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# Descargamos dataset de https://archive.ics.uci.edu/dataset/2/adult?utm_source=chatgpt.com

from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
adult = fetch_ucirepo(id=2) 
  
# data (as pandas dataframes) 
X = adult.data.features 
y = adult.data.targets 
  
# metadata 
print(adult.metadata) 
  
# variable information 
print(adult.variables) 


{'uci_id': 2, 'name': 'Adult', 'repository_url': 'https://archive.ics.uci.edu/dataset/2/adult', 'data_url': 'https://archive.ics.uci.edu/static/public/2/data.csv', 'abstract': 'Predict whether annual income of an individual exceeds $50K/yr based on census data. Also known as "Census Income" dataset. ', 'area': 'Social Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 48842, 'num_features': 14, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Income', 'Education Level', 'Other', 'Race', 'Sex'], 'target_col': ['income'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 1996, 'last_updated': 'Tue Sep 24 2024', 'dataset_doi': '10.24432/C5XW20', 'creators': ['Barry Becker', 'Ronny Kohavi'], 'intro_paper': None, 'additional_info': {'summary': "Extraction was done by Barry Becker from the 1994 Census database.  A set of reasonably clean records was extracted using the fol

In [29]:
pd.set_option('display.float_format', '{:.2f}'.format)

In [30]:
# Visualizamos las primeras 15 filas

df = pd.concat([X, y], axis=1)
display(df.sample(5))

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
7370,42,Private,166740,Bachelors,13,Divorced,Prof-specialty,Not-in-family,White,Male,0,0,40,United-States,<=50K
3459,21,?,356772,HS-grad,9,Never-married,?,Unmarried,White,Female,0,0,40,United-States,<=50K
2374,55,Private,67450,Doctorate,16,Never-married,Prof-specialty,Not-in-family,White,Male,0,0,40,England,<=50K
12024,67,Private,64148,Some-college,10,Divorced,Other-service,Unmarried,Black,Female,0,0,41,United-States,<=50K
18216,54,Private,118793,HS-grad,9,Married-civ-spouse,Adm-clerical,Husband,White,Male,0,0,45,United-States,<=50K


In [31]:
# Dimensiones del dataset

print(f"El dataset contiene {df.shape[0]} filas y {df.shape[1]} columnas.")    # cantidad de filas y columnas

El dataset contiene 48842 filas y 15 columnas.


In [32]:
# Nombres de las columnas del dataset
df.columns.tolist()

['age',
 'workclass',
 'fnlwgt',
 'education',
 'education-num',
 'marital-status',
 'occupation',
 'relationship',
 'race',
 'sex',
 'capital-gain',
 'capital-loss',
 'hours-per-week',
 'native-country',
 'income']

In [33]:
# Información general y tipos de datos
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             48842 non-null  int64 
 1   workclass       47879 non-null  object
 2   fnlwgt          48842 non-null  int64 
 3   education       48842 non-null  object
 4   education-num   48842 non-null  int64 
 5   marital-status  48842 non-null  object
 6   occupation      47876 non-null  object
 7   relationship    48842 non-null  object
 8   race            48842 non-null  object
 9   sex             48842 non-null  object
 10  capital-gain    48842 non-null  int64 
 11  capital-loss    48842 non-null  int64 
 12  hours-per-week  48842 non-null  int64 
 13  native-country  48568 non-null  object
 14  income          48842 non-null  object
dtypes: int64(6), object(9)
memory usage: 5.6+ MB


## 1.2 Valores Nulos

Se valida la existencia de valores nulos, revisamos tanto el formato del valor nulo convertido mediante repositorio **ucimlrepo** como también su formato original.

Detectamos que la librería de origen no normalizó el símbolo de faltante de forma consistente, por lo que unificamos ambas codificaciones antes de continuar.

Revisaremos la existencia de los siguientes símbolos que representan en este dataset valores nulos ó faltantes: "NaN", "?" y " ".

In [34]:
# Cantidad de nulos por columna
df.isnull().sum()

# Solo las columnas que sí tienen nulos, con porcentaje
nulos = df.isnull().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
pd.DataFrame({
    'Valores faltantes': nulos,
    'Porcentaje (%)': (nulos / len(df) * 100).round(2)
})

,Valores faltantes,Porcentaje (%)
occupation,966,1.98
workclass,963,1.97
native-country,274,0.56


In [35]:
# Verificación de nulos en formato original de dataset.

(df == '?').sum()[lambda x: x > 0]

workclass         1836
occupation        1843
native-country     583
dtype: int64

In [36]:
# Busca strings vacíos o que son solo espacios en las columnas de texto
cols_obj = df.select_dtypes(include='object').columns

vacios = {}
for col in cols_obj:
    n = (df[col].astype(str).str.strip() == '').sum()
    if n > 0:
        vacios[col] = n

print('Columnas con celdas vacías/solo espacios:', vacios or 'ninguna')

Columnas con celdas vacías/solo espacios: ninguna


In [37]:
nulos_nan = df.isnull().sum()
nulos_signo = (df == '?').sum()
total_faltantes = nulos_nan.add(nulos_signo, fill_value=0)
total_faltantes = total_faltantes[total_faltantes > 0].sort_values(ascending=False)
pd.DataFrame({
    'Valores faltantes': total_faltantes.astype(int),
    'Porcentaje (%)': (total_faltantes / len(df) * 100).round(2)
})

,Valores faltantes,Porcentaje (%)
occupation,2809,5.75
workclass,2799,5.73
native-country,857,1.75


In [38]:
pd.set_option('display.max_columns', None)

df[df[['workclass','occupation','native-country']].isnull().any(axis=1) |
   (df[['workclass','occupation','native-country']] == '?').any(axis=1)].head(5)

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
14,40,Private,121772,Assoc-voc,11,Married-civ-spouse,Craft-repair,Husband,Asian-Pac-Islander,Male,0,0,40,?,>50K
27,54,?,180211,Some-college,10,Married-civ-spouse,?,Husband,Asian-Pac-Islander,Male,0,0,60,South,>50K
38,31,Private,84154,Some-college,10,Married-civ-spouse,Sales,Husband,White,Male,0,0,38,?,>50K
51,18,Private,226956,HS-grad,9,Never-married,Other-service,Own-child,White,Female,0,0,30,?,<=50K
61,32,?,293936,7th-8th,4,Married-spouse-absent,?,Not-in-family,White,Male,0,0,40,?,<=50K


### Hallazgos de valores faltantes: 

Se encuentran 6,465 celdas con valores nulos. Sin embargo por categoría los valores nulos representan aproximadamente entre un 5% y un 6% para las columnas de "occupation" y "worldclass", y para la columna "native-country" este representa apenas un 1.75%.   

## 1.3 Imputación

Debido a los valores faltantes representan aproximadamente un 5% aproximadamente ó inclusive un valor menor del total de cada columna y a la vez al tratarse de datos de tipo cualitativos, **imputaremos con base en la moda**.

In [39]:
cols_con_nulos = ['workclass', 'occupation', 'native-country']

# Moda de cada columna (ignora nulos y "?")
modas = {}
for col in cols_con_nulos:
    moda = df.loc[~df[col].isnull() & (df[col] != '?'), col].mode()[0]
    modas[col] = moda
    print(f'{col}: moda = {moda}')

workclass: moda = Private
occupation: moda = Prof-specialty
native-country: moda = United-States


In [40]:
# Imputación: reemplaza NaN y "?" por la moda de cada columna
df_clean = df.copy()

for col in cols_con_nulos:
    df_clean[col] = df_clean[col].replace('?', modas[col])
    df_clean[col] = df_clean[col].fillna(modas[col])

# Verificación
print(df_clean[cols_con_nulos].isnull().sum())
print((df_clean[cols_con_nulos] == '?').sum())

workclass         0
occupation        0
native-country    0
dtype: int64
workclass         0
occupation        0
native-country    0
dtype: int64


Imputacion de punto (.) al final del valor en columna "income".

In [41]:
# Quita el punto final para unificar "<=50K." con "<=50K" y ">50K." con ">50K"
df['income'] = df['income'].str.rstrip('.')

# Verificación
df['income'].unique()

array(['<=50K', '>50K'], dtype=object)

In [42]:
df_clean['income'] = df_clean['income'].str.rstrip('.')
df_clean['income'].unique()

array(['<=50K', '>50K'], dtype=object)

### Hallazgos de Imputación:

Se imputan los datos nulos identificados tanto con simbolo "NaN" como con simbolo "?", dejando en cero la cantidad de datos nulos.

También se limpia un punto (.) en la columna "income" reduciendo la cantidad de categorías para esa variable a dos únicamente.

# 2. Análisis Exploratorio de Datos   

Selección de variables cuantitativas:  

           
0   age              
2   fnlwgt           
4   education-num    
10  capital-gain     
11  capital-loss     
12  hours-per-week   


In [43]:
# Lista de variables cuantitativas
vars_cuantitativas = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']

# Tabla resumen con estadísticas descriptivas de todas juntas
df[vars_cuantitativas].describe()

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
count,48842.00,48842.00,48842.00,48842.00,48842.00,48842.00
mean,38.64,189664.13,10.08,1079.07,87.50,40.42
std,13.71,105604.03,2.57,7452.02,403.00,12.39
min,17.00,12285.00,1.00,0.00,0.00,1.00
25%,28.00,117550.50,9.00,0.00,0.00,40.00
50%,37.00,178144.50,10.00,0.00,0.00,40.00
75%,48.00,237642.00,12.00,0.00,0.00,45.00
max,90.00,1490400.00,16.00,99999.00,4356.00,99.00


In [44]:
resumen = df[vars_cuantitativas].describe().T
resumen['rango'] = resumen['max'] - resumen['min']
resumen['mediana'] = df[vars_cuantitativas].median()
resumen

,count,mean,std,min,25%,50%,75%,max,rango,mediana
age,48842.00,38.64,13.71,17.00,28.00,37.00,48.00,90.00,73.00,37.00
fnlwgt,48842.00,189664.13,105604.03,12285.00,117550.50,178144.50,237642.00,1490400.00,1478115.00,178144.50
education-num,48842.00,10.08,2.57,1.00,9.00,10.00,12.00,16.00,15.00,10.00
capital-gain,48842.00,1079.07,7452.02,0.00,0.00,0.00,0.00,99999.00,99999.00,0.00
capital-loss,48842.00,87.50,403.00,0.00,0.00,0.00,0.00,4356.00,4356.00,0.00
hours-per-week,48842.00,40.42,12.39,1.00,40.00,40.00,45.00,99.00,98.00,40.00


## Hallazgos del análisis estadístico descriptivo



